# Export existing screen images for review

Use a CPU runtime. No token, model download or inference required. This notebook checks manifest hashes and downloads the 48 existing images and their metadata from four comparison methods. Attach the ZIP in chat. It does not change your results.

In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import drive, files
import json, hashlib

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/KLTN_OCSD')
RUN_ID = '20261028T225607Z_4480ea4d'
manifest_path = ROOT / 'results' / 'runs' / RUN_ID / 'run_manifest.json'
manifest = json.loads(manifest_path.read_text())
methods = {'gligen', 'gligen_bon', 'grounded_repair', 'grounded_repair_no_m2'}
selected = []
for relative, digest in manifest['artifacts'].items():
    rel = Path(relative)
    if (rel.parts[:3] == ('outputs', 'runs', RUN_ID)
            and len(rel.parts) == 6 and rel.parts[3] == 'quickdraw'
            and rel.parts[4] in methods and rel.suffix in {'.png', '.json'}):
        p = (ROOT / rel).resolve()
        assert p.is_relative_to(ROOT.resolve())
        assert p.is_file(), f'Missing: {p}'
        assert hashlib.sha256(p.read_bytes()).hexdigest() == digest, f'Changed: {p}'
        selected.append((p, relative))
assert len(selected) == 96, f'Expected 48 images and 48 metadata files, found {len(selected)}'
archive = '/content/ocsd_visual_review.zip'
with ZipFile(archive, 'w', ZIP_DEFLATED) as z:
    z.write(manifest_path, 'run_manifest.json')
    for p, relative in selected:
        z.write(p, relative)
print('Exported existing images and repair metadata; no model execution:', len(selected))
files.download(archive)
